In [1]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)

if device == 'cuda':
    torch.cuda.manual_seed_all(777)
    print('cuda')

In [3]:
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [5]:
mnist_train = dsets.MNIST(root='MNIST_data/',
                          train = True,
                          transform = transforms.ToTensor(),
                          download = True
                          )
mnist_test = dsets.MNIST(root ='MNIST_data/',
                         train = False,
                         transform = transforms.ToTensor(),
                         download = True
                         )

100%|██████████| 9.91M/9.91M [00:00<00:00, 61.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.72MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.1MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.09MB/s]


In [7]:
# 다운받은 데이터셋을 나누기위한 과정 - 데이터로더
data_loader = torch.utils.data.DataLoader(dataset = mnist_train,
                                          batch_size = batch_size,
                                          shuffle = True,
                                          drop_last = True)

In [8]:
class CNN(torch.nn.Module):

    def __init__(self):
        super(CNN, self).__init__()
        
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size= 3, stride= 1 , padding= 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size= 2, stride= 2)

        )

        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size= 3, stride= 1, padding= 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size= 3, stride= 2)
        )

        self.fc = torch.nn.Linear(7*7*64, 10, bias= True)

        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [18]:
print(out.shape)

NameError: name 'out' is not defined

In [16]:
model = CNN().to(device)

In [11]:
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [13]:
total_batch = len(data_loader)
print('배치의 수 : {}'.format(total_batch))

배치의 수 : 600


In [17]:
for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad() # 기울기 초기화
        hypothesis = model(X) # 모델을 통해 예측값 계산
        cost = criterion(hypothesis, Y) # 예측값과 실제값 Y의 손실 계산
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch 


    print('[Epoch : {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (100x2304 and 3136x10)